# Silverwing-ML — Production Training Pipeline (Free GPU)

Manufacturing run for the 102M Silverwing decoder on **real data** (FineWeb-Edu + Wikipedia),
with fp16 mixed precision and **Google Drive persistence** so Colab disconnects never lose progress.

**Before you start:**
1. `Runtime` > `Change runtime type` > **T4 GPU** (free tier).
2. Run cells **1-6 in order** (setup + corpus + tokenizer, ~1-2 h first time, then cached on Drive).
3. Run **Cell 7 (Pretrain)**. It resumes from Drive automatically. If the session dies or you
   need to stop (`Runtime > Interrupt execution`), just **run Cell 7 again** — at most ~15 min lost.
4. When pretrain reports COMPLETE, run cells 8-12 (SFT + evaluation).

**Budget math (T4, fp16):** ~6-10K tok/s -> 36,000 steps x 8,192 tok/step = ~295M tokens
= roughly 3-5 free sessions. Kaggle (30 h/week free) works too with minor path changes.

**Second free GPU source:** [Kaggle Notebooks](https://www.kaggle.com/code) — P100/T4x2,
30 hrs/week. Upload this notebook, enable GPU + Internet in settings.

In [ ]:
# Cell 1: Mount Google Drive (all progress persists here)
import os

from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'
for sub in ['checkpoints/pretrain', 'checkpoints/sft', 'corpus', 'tokenizer']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print('Drive ready:', DRIVE)

In [ ]:
# Cell 2: Install dependencies + verify GPU
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy datasets huggingface_hub
import torch

assert torch.cuda.is_available(), 'NO GPU - Runtime > Change runtime type > T4 GPU'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)')
print(f'torch {torch.__version__}, fp16 tensor cores: OK')

In [ ]:
# Cell 3: Clone the repo (public)
import os

if not os.path.exists('/content/Silverwing-ML'):
    !git clone https://github.com/oledesug-source/silverwing-ml.git /content/Silverwing-ML
os.chdir('/content/Silverwing-ML')
!git pull --ff-only || echo 'already latest'
!git log --oneline -1

In [ ]:
# Cell 4: Real pretraining corpus (FineWeb-Edu 400K + Wikipedia 100K)
# Cached on Drive: a finished corpus is restored instantly; a partial raw
# download is resumed via --skip-existing instead of re-downloading.
import shutil
import subprocess
from pathlib import Path

CORPUS = Path('experiments/corpus-external')
RAW = Path('experiments/ingested')
D_CORPUS = Path(DRIVE) / 'corpus/corpus-external'
D_RAW = Path(DRIVE) / 'corpus/raw-ingested'

if D_CORPUS.exists() and any(D_CORPUS.glob('train.*.jsonl')):
    if CORPUS.exists():
        shutil.rmtree(CORPUS)
    shutil.copytree(D_CORPUS, CORPUS)
    print('Corpus restored from Drive:', sorted(p.name for p in CORPUS.glob('*.jsonl')))
else:
    if D_RAW.exists() and not RAW.exists():
        shutil.copytree(D_RAW, RAW)
        print('Raw ingest restored from Drive')
    r = subprocess.run(
        ['python', 'scripts/ingest_external_v2.py', '--preset', 'all', '--skip-existing'],
        capture_output=True, text=True,
    )
    print(r.stdout[-3000:])
    print(r.stderr[-2000:])
    assert r.returncode == 0, 'corpus build failed - see output above'
    for src, dst in [(CORPUS, D_CORPUS), (RAW, D_RAW)]:
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    print('Corpus built and synced to Drive')

train_shard = next(CORPUS.glob('train.*.jsonl'))
n_docs = sum(1 for _ in open(train_shard, encoding='utf-8'))
print(f'train documents: {n_docs:,}')

In [ ]:
# Cell 5: Tokenizer v2 trained on the real corpus (falls back to committed tokenizer)
import json
import shutil
import subprocess
from pathlib import Path

TOK = Path('experiments/tokenizer-v2')
D_TOK = Path(DRIVE) / 'tokenizer/tokenizer-v2'

try:
    if D_TOK.exists() and (D_TOK / 'vocab.json').exists():
        if TOK.exists():
            shutil.rmtree(TOK)
        shutil.copytree(D_TOK, TOK)
        print('Tokenizer v2 restored from Drive')
    else:
        r = subprocess.run(
            ['python', 'scripts/train_tokenizer.py',
             '--corpus-dir', 'experiments/corpus-external',
             '--vocab-size', '16384',
             '--max-documents', '60000',
             '--output-dir', 'experiments/tokenizer-v2'],
            capture_output=True, text=True,
        )
        print(r.stdout[-1500:], r.stderr[-1500:])
        assert r.returncode == 0, 'tokenizer training failed'
        if D_TOK.exists():
            shutil.rmtree(D_TOK)
        shutil.copytree(TOK, D_TOK)
        print('Tokenizer v2 trained and synced to Drive')
    TOKENIZER_DIR = 'experiments/tokenizer-v2'
except Exception as e:
    print(f'WARNING: tokenizer v2 unavailable ({e}); using committed tokenizer')
    TOKENIZER_DIR = 'experiments/tokenizer'

print('TOKENIZER_DIR =', TOKENIZER_DIR)

In [ ]:
# Cell 6: Production pretraining config (fp16 AMP, Drive checkpoints)
# NOTE: max_steps is FIXED at 36000 forever after first run - the trainer's
# resume guard rejects config drift, so do not change hyperparams mid-run.
import yaml

cfg = {'training': {
    'version': 'training-v1',
    'model_config_path': 'configs/model.yaml',
    'corpus_dir': 'experiments/corpus-external',
    'tokenizer_dir': TOKENIZER_DIR,
    'checkpoint_dir': f'{DRIVE}/checkpoints/pretrain',
    'batch_size': 16,
    'grad_accum_steps': 1,
    'block_size': 512,
    'max_steps': 36000,
    'warmup_steps': 2000,
    'lr': 3.0e-4,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.1,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 50,
    'eval_steps': 1000,
    'eval_sequences': 16,
    'save_steps': 1000,
    'verify_dataset': False,
    'expected_dataset_hash': None,
    'require_validation': True,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/training_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print('configs/training_production.yaml written')
print(f'target: {cfg["training"]["max_steps"]:,} steps x {cfg["training"]["batch_size"] * cfg["training"]["block_size"]:,} tokens = {cfg["training"]["max_steps"] * cfg["training"]["batch_size"] * cfg["training"]["block_size"] / 1e6:.0f}M tokens')

In [ ]:
# Cell 7: PRETRAIN RUNNER - safe to re-run any number of times.
# Resumes from the newest Drive checkpoint and trains to step 36000.
# Session died? Runtime > Interrupt needed? Just run this cell again.
import re
import subprocess
import time
from pathlib import Path

CKPT = Path(DRIVE) / 'checkpoints/pretrain'
TARGET = 36000

def latest_step(d: Path) -> int:
    steps = []
    for p in d.glob('step-*.pt'):
        m = re.match(r'step-(\d+)\.pt', p.name)
        if m:
            steps.append(int(m.group(1)))
    return max(steps) if steps else 0

done = latest_step(CKPT)
if done >= TARGET:
    print(f'PRETRAIN COMPLETE ({done:,} steps). Continue to Cell 8.')
else:
    cks = sorted(CKPT.glob('step-*.pt'), key=lambda p: int(re.sub(r'\D', '', p.name)))
    cmd = ['python', 'scripts/train.py',
           '--config', 'configs/training_production.yaml',
           '--device', 'cuda',
           '--no-clean-repo-check']
    if cks:
        cmd += ['--resume-from', str(cks[-1])]
        print(f'Resuming from step {done:,} / {TARGET:,}')
    else:
        print('Fresh pretraining start')
    t0 = time.time()
    r = subprocess.run(cmd)
    print(f'exit={r.returncode} elapsed={(time.time() - t0) / 60:.1f} min')
    print(f'progress: {latest_step(CKPT):,} / {TARGET:,} steps')
    if latest_step(CKPT) < TARGET:
        print('Session ended early? Run this cell again to continue.')

In [ ]:
# Cell 8: Verify SFT dataset (committed in repo)
from pathlib import Path

sft_ds = Path('experiments/sft/sft-v2-all.jsonl')
assert sft_ds.exists(), 'SFT dataset missing from repo - check git clone'
n = sum(1 for _ in open(sft_ds, encoding='utf-8'))
print(f'SFT examples: {n:,}')

In [ ]:
# Cell 9: SFT run (fp16 AMP, init from pretrained best, Drive checkpoints)
import subprocess
import yaml

PRETRAIN_BEST = f'{DRIVE}/checkpoints/pretrain/best.pt'
assert __import__('os').path.exists(PRETRAIN_BEST), 'pretrain best.pt not found - finish Cell 7 first'

cfg = {'sft': {
    'version': 'sft-v1',
    'model_config_path': 'configs/model.yaml',
    'tokenizer_dir': TOKENIZER_DIR,
    'init_from': PRETRAIN_BEST,
    'dataset_path': 'experiments/sft/sft-v2-all.jsonl',
    'checkpoint_dir': f'{DRIVE}/checkpoints/sft',
    'batch_size': 16,
    'block_size': 512,
    'max_steps': 2000,
    'warmup_steps': 100,
    'lr': 1.0e-4,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.1,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 25,
    'eval_steps': 200,
    'eval_examples': 64,
    'save_steps': 500,
    'eval_fraction': 0.05,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/sft_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
subprocess.run(['python', 'scripts/train_sft.py',
                '--config', 'configs/sft_production.yaml',
                '--device', 'cuda',
                '--no-clean-repo-check'])

In [ ]:
# Cell 10: Generation test
import sys

import torch

sys.path.insert(0, '.')
from foundation.inference import Generator
from foundation.model.config import ModelConfig
from foundation.model.model import SilverwingDecoder
from foundation.tokenizer import TokenizerV2

tok = TokenizerV2.load(TOKENIZER_DIR)
ckpt_path = f'{DRIVE}/checkpoints/sft/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cfg = ModelConfig.from_yaml('configs/model.yaml')
model = SilverwingDecoder(model_cfg)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(state.get('model_state', state))
model = model.to(device).eval()
print(f'Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params from {ckpt_path}')

gen = Generator(model, tok)
prompts = [
    'What is 2 + 2? ',
    'The answer to 3 * 3 is ',
    'Solve for x: 2x + 5 = 13. Step 1:',
    'The capital of France is ',
]
for p in prompts:
    result = gen.generate(p, max_new_tokens=80, temperature=0.7, top_k=50)
    print(f'--- {p!r}\n{result}')

In [ ]:
# Cell 11: Math benchmark against regression gates
import subprocess

subprocess.run(['python', 'scripts/run_benchmark.py',
                '--benchmark', 'math-benchmark-v1',
                '--model', f'silverwing:{DRIVE}/checkpoints/sft/best.pt',
                '--tokenizer-dir', TOKENIZER_DIR])

In [ ]:
# Cell 12: Manufacturing summary
import json
from pathlib import Path

for name, report in [('pretrain', Path(DRIVE) / 'checkpoints/pretrain/training_report.json'),
                     ('sft', Path(DRIVE) / 'checkpoints/sft/sft_report.json')]:
    if report.exists():
        d = json.loads(report.read_text(encoding='utf-8'))
        print(f'[{name}] eval_loss={d.get("final_eval_loss")} ppl={d.get("final_perplexity")} '
              f'best={d.get("best_eval_loss")} tok_seen~{d.get("tokens_seen", 0):,}')
    else:
        print(f'[{name}] report not found yet')
print('\nArtifacts on Drive:')
for p in sorted(Path(DRIVE).rglob('*.pt')):
    print(f'  {p.relative_to(DRIVE)} ({p.stat().st_size / 1e9:.2f} GB)')
print('\nNext: download sft/best.pt to your machine and serve it through the platform:')
print('  python scripts/serve_platform.py')